# Kaggle GPU End-to-End Runner
Runs the modern-backbone (Nemotron-Mini-4B + Phi-3.5-mini) intent/slot experiment matrix end to end on **Kaggle GPU**, using packaged local datasets and copied training code.


## Pre-flight Checklist (READ FIRST)

Before running, make sure:

1. **GPU is ON** — Notebook settings -> Accelerator -> GPU (T4/P100).
2. **Internet is ON** — Notebook settings -> Internet -> On. Needed only if you let models download from Hugging Face (see Model Configuration). Datasets are all packaged locally (including CLINC150), so data needs no internet.
3. **Models are attached** — the two ~4B backbones are NOT in this pack (too big). Either:
   - Attach them as Kaggle Datasets and point the paths in **Model Configuration**, or
   - Leave the Hugging Face IDs and let them download (Internet ON).
4. This pack folder is attached/available so `code/` and `datasets/` resolve.

Expected outputs: per-run JSON under `code/results/*_usmodern_*.json` and a consolidated `kaggle_run_summary.csv`.


## Environment Setup
Install dependencies, verify CUDA, and wire notebook paths.


In [ ]:
import os, sys, json, subprocess
from pathlib import Path

ROOT = Path.cwd()
PACK_DIR = ROOT if (ROOT / 'code').exists() else ROOT / 'kaggle_gpu_pack'
CODE_DIR = PACK_DIR / 'code'
DATA_DIR = PACK_DIR / 'datasets' / 'raw'
RESULTS_DIR = CODE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('PACK_DIR:', PACK_DIR)
print('CODE_DIR:', CODE_DIR)
print('DATA_DIR:', DATA_DIR)


In [ ]:
# Kaggle usually already has torch; install/upgrade the rest quietly
%pip -q install -U transformers datasets accelerate sentencepiece scikit-learn pandas


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected -- enable GPU in notebook settings for reasonable runtimes.')


## Dataset Staging
Copy packaged datasets into `code/data/raw` where the scripts expect them. CLINC150 is packaged locally too, so this is fully offline.


In [ ]:
import shutil
TARGET_RAW = CODE_DIR / 'data' / 'raw'
TARGET_RAW.mkdir(parents=True, exist_ok=True)

for ds in ['atis_iob', 'banking77', 'snips', 'massive', 'clinc150']:
    src = DATA_DIR / ds
    if not src.exists():
        print('[warn] missing packaged dataset:', ds)
        continue
    dst = TARGET_RAW / ds
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

print('Staged datasets:', sorted([p.name for p in TARGET_RAW.iterdir()]))
print('CLINC150 local file present:', (TARGET_RAW / 'clinc150' / 'data_full.json').exists())


## Model Configuration
Set model paths. Prefer Kaggle Dataset paths if you uploaded the model folders; otherwise the Hugging Face IDs are used (Internet ON).


In [ ]:
MODEL_PATHS = {
    'nemotron': '/kaggle/input/nemotron-mini-4b',  # or 'nvidia/Nemotron-Mini-4B-Instruct'
    'phi': '/kaggle/input/phi-3.5-mini',           # or 'microsoft/Phi-3.5-mini-instruct'
}

if not Path(MODEL_PATHS['nemotron']).exists():
    MODEL_PATHS['nemotron'] = 'nvidia/Nemotron-Mini-4B-Instruct'
if not Path(MODEL_PATHS['phi']).exists():
    MODEL_PATHS['phi'] = 'microsoft/Phi-3.5-mini-instruct'

MODEL_PATHS


## Run Training Matrix
Runs probe sweep + pruned d3 + pruned d12 across **all 5 datasets** (ATIS, SNIPS, MASSIVE, CLINC150, BANKING77) for both models.

**Publication-grade budget**: training size scales with intent cardinality (`max_train = max(400, 20 x num_intents)`) so high-cardinality datasets (CLINC150=150, BANKING77=77, MASSIVE=60) are no longer starved. Resumable via `run_if_missing`.


In [ ]:
import shlex

os.chdir(CODE_DIR)

# intent cardinality per dataset (drives publication-grade budget scaling)
INTENTS = {'atis': 17, 'snips': 7, 'massive': 60, 'clinc150': 150, 'banking77': 77}

DATASETS = ['atis', 'snips', 'massive', 'clinc150', 'banking77']
MODEL_ORDER = [('nemotron', MODEL_PATHS['nemotron']), ('phi', MODEL_PATHS['phi'])]

# --- publication-grade knobs (the reason to use a GPU) ---
PRUNED_EPOCHS = 3
BATCH_SIZE = 16
PROBE_EPOCHS = 50

def pruned_max_train(ds):
    # >= 20x intents, floored so small-label datasets still get enough data
    return max(400, 20 * INTENTS[ds])

def probe_max_train(ds):
    # a bit larger for the cheap probe; capped to keep it fast
    return min(3000, max(600, 40 * INTENTS[ds]))

def run(cmd):
    print('\n[RUN]', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

def run_if_missing(output_path, cmd):
    if Path(output_path).exists():
        print('[SKIP]', output_path)
    else:
        run(cmd)

for model_tag, model_path in MODEL_ORDER:
    for ds in DATASETS:
        pmt = pruned_max_train(ds)
        probemt = probe_max_train(ds)

        out_probe = f'results/{ds}_probe_sweep_usmodern_{model_tag}.json'
        out_d3 = f'results/{ds}_pruned_depth3_usmodern_{model_tag}.json'
        out_d12 = f'results/{ds}_pruned_depth12_usmodern_{model_tag}.json'

        probe_cmd = (
            f'PYTHONPATH=src python scripts/probe_sweep.py --dataset {ds} '
            f'--model-path {shlex.quote(model_path)} --max-train {probemt} '
            f'--probe-epochs {PROBE_EPOCHS} --output {out_probe}'
        )
        d3_cmd = (
            f'PYTHONPATH=src python scripts/train_pruned.py --dataset {ds} --depth 3 '
            f'--model-path {shlex.quote(model_path)} --epochs {PRUNED_EPOCHS} --batch-size {BATCH_SIZE} '
            f'--max-train {pmt} --output {out_d3}'
        )
        d12_cmd = (
            f'PYTHONPATH=src python scripts/train_pruned.py --dataset {ds} --depth 12 '
            f'--model-path {shlex.quote(model_path)} --epochs {PRUNED_EPOCHS} --batch-size {BATCH_SIZE} '
            f'--max-train {pmt} --output {out_d12}'
        )

        print(f'== {model_tag} :: {ds} :: intents={INTENTS[ds]} pruned_max_train={pmt} probe_max_train={probemt} ==')
        run_if_missing(out_probe, probe_cmd)
        run_if_missing(out_d3, d3_cmd)
        run_if_missing(out_d12, d12_cmd)

print('\nMatrix run complete.')


## Results Summary
Load all generated JSON outputs and build a compact summary table + CSV.


In [ ]:
import pandas as pd

rows = []
for p in sorted((CODE_DIR / 'results').glob('*_usmodern_*.json')):
    j = json.loads(p.read_text())
    rows.append({
        'file': p.name,
        'dataset': j.get('dataset'),
        'kind': j.get('kind'),
        'depth': j.get('depth'),
        'model': j.get('model'),
        'n_train': j.get('n_train'),
        'n_intents': j.get('n_intents'),
        'intent_accuracy': j.get('intent_accuracy'),
        'slot_f1': (j.get('slot_f1') or {}).get('f1') if isinstance(j.get('slot_f1'), dict) else None,
        'train_time_s': j.get('train_time_s'),
        'recommended_depth': j.get('recommended_depth'),
    })

df = pd.DataFrame(rows).sort_values(['dataset', 'model', 'kind', 'depth'], na_position='last')
df.head(60)


In [ ]:
summary_path = PACK_DIR / 'kaggle_run_summary.csv'
df.to_csv(summary_path, index=False)
print('Saved:', summary_path)
print('Artifacts:', len(list((CODE_DIR / 'results').glob('*_usmodern_*.json'))))
